**Roll No:** b23bb1025

**Approach:**
1. **Stage 1:** Distill CLIP ViT-L/14 image features into the existing B23BB1025 backbone (35 epochs, AdamW)
2. **Stage 2:** Fine-tune the same backbone for binary age classification with MixUp, EMA, differential LRs, and strong augmentation (30 epochs)

****Config Setup****

In [ ]:
import os
import csv
import math
from torchvision import transforms
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

data_root = "dataset/"
train_path = os.path.join(data_root, "train")
val_path = os.path.join(data_root, "valid")
val_labels = os.path.join(data_root, "valid_labels.csv")

img_size = 224
bsz = 64
n_workers = 4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)   # reproducibility
print(f"using: {device}")

# clip config
clip_model_name = "ViT-L/14"
clip_feat_dim = 768

# reuse the exact model architecture already defined for submission
# no pretrained weights, no ResNet-50
from B23BB1025 import build_model

using: cuda


****Gather Clip Embeddings for Stage 1 Distillation****

In [2]:
def get_clip_embeddings(train_root, val_dir, val_csv):
    """grab clip features for ALL images (train + val) for distillation"""
    import clip
    print("loading clip...")

    model, preproc = clip.load(clip_model_name, device=device)
    model.eval()

    feats = []

    # train images first (class 0 then class 1)
    for cls in [0, 1]:
        folder = os.path.join(train_root, str(cls))
        files = sorted([f for f in os.listdir(folder) if f.endswith(".png")])
        print(f"  train class {cls}: {len(files)} images")

        for i in range(0, len(files), 32):
            chunk = files[i:i+32]
            imgs = [preproc(Image.open(os.path.join(folder, f)).convert("RGB")) for f in chunk]
            batch = torch.stack(imgs).to(device)
            with torch.no_grad():
                emb = F.normalize(model.encode_image(batch).float(), dim=-1)
            feats.append(emb.cpu())

    # val images too since we're training on everything for final submission
    with open(val_csv) as f:
        val_files = [r["image"] for r in csv.DictReader(f)]
    print(f"  val: {len(val_files)} images")

    for i in range(0, len(val_files), 32):
        chunk = val_files[i:i+32]
        imgs = [preproc(Image.open(os.path.join(val_dir, f)).convert("RGB")) for f in chunk]
        batch = torch.stack(imgs).to(device)
        with torch.no_grad():
            emb = F.normalize(model.encode_image(batch).float(), dim=-1)
        feats.append(emb.cpu())

    all_feats = torch.cat(feats, dim=0)
    print(f"total features: {all_feats.shape}")

    del model
    torch.cuda.empty_cache()
    return all_feats

****DataSet Loader Setup****

In [ ]:
# combined dataset - train + val together, returns index for clip feature lookup
class FaceFullSet(Dataset):
    def __init__(self, train_root, val_dir, val_csv, tfm=None):
        self.tfm = tfm
        self.data = []

        # add all train images
        for lbl in [0, 1]:
            d = os.path.join(train_root, str(lbl))
            for fn in sorted(os.listdir(d)):
                if fn.endswith(".png"):
                    self.data.append((os.path.join(d, fn), lbl))

        # add val images with their labels
        with open(val_csv) as f:
            for r in csv.DictReader(f):
                self.data.append((os.path.join(val_dir, r["image"]), int(r["label"])))

        print(f"combined dataset: {len(self.data)} total images")

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        path, lbl = self.data[idx]
        img = Image.open(path).convert("RGB")
        if self.tfm: img = self.tfm(img)
        return img, lbl, idx

In [ ]:
# stage 1 - minimal augmentation for CLIP distillation
aug_s1 = transforms.Compose([
    transforms.RandomResizedCrop(img_size, scale=(0.65, 1.0)),  # slightly wider crop range
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),       # mild colour shift
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# stage 2 - aggressive augmentation for fine-tuning
aug_s2 = transforms.Compose([
    transforms.RandomResizedCrop(img_size, scale=(0.5, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.4, 0.4, 0.4, 0.1),
    transforms.RandomGrayscale(p=0.1),
    transforms.RandomRotation(15),
    transforms.RandAugment(num_ops=2, magnitude=9),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.15)),
])

In [ ]:
def lr_schedule(ep, warmup, peak_lr, total):
    """warmup then cosine decay"""
    if ep < warmup:
        return peak_lr * (ep + 1) / warmup
    t = (ep - warmup) / (total - warmup)
    return 1e-6 + 0.5 * (peak_lr - 1e-6) * (1 + math.cos(math.pi * t))

# ---- MixUp helpers ----
def mixup_data(x, y, alpha=0.3):
    lam = float(np.random.beta(alpha, alpha)) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam

def mixup_loss(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

---
## check if we already have a distilled backbone 
(Reason Being if in future i do subsequent runs to tweak Hyperparms i dont have to start from the beginning)

In [ ]:
backbone_file = "backbone_b23bb1025.pth"   # distilled checkpoint for existing backbone

if os.path.exists(backbone_file):
    print(f"found {backbone_file}, loading it up")
    net = torch.load(backbone_file, map_location=device, weights_only=False)
    net.to(device)
    with torch.no_grad():
        feat_dim = net.extract_features(torch.zeros(1, 3, img_size, img_size, device=device)).shape[1]
    if feat_dim != 512:
        raise RuntimeError(f"Unexpected backbone feature dim: {feat_dim}. Expected 512 (non-ResNet50 setup).")
    print(f"params: {sum(p.numel() for p in net.parameters()):,}")
    print("backbone loaded - skip the next cell and go straight to stage 2")
else:
    print(f"{backbone_file} not found, need to run stage 1 distillation first")
    print("run the next cell to start stage 1")

found backbone_pretrain.pth, loading it up
params: 11,325,058
backbone loaded - skip the next cell and go straight to stage 2


## Stage 1: CLIP Feature Distillation (existing B23BB1025 backbone)

Teach the existing backbone to mimic CLIP ViT-L/14 image features via cosine-similarity loss.

Key points:
- No pretrained weights
- No ResNet-50
- Uses the same architecture as `B23BB1025.py`
- Projection head: 512 -> 1024 -> 768 (backbone -> CLIP ViT-L/14 feature dims)
- AdamW optimiser, peak LR 1e-3, **35 epochs**
- 5-epoch warmup + cosine decay

Skip this cell if `backbone_b23bb1025.pth` already exists.

In [ ]:
# pull clip features for all images (train + val)
clip_feats = get_clip_embeddings(train_path, val_path, val_labels)

# fresh model using existing B23BB1025 architecture (no pretrained, no ResNet-50)
net = build_model(num_classes=2).to(device)
with torch.no_grad():
    feat_dim = net.extract_features(torch.zeros(1, 3, img_size, img_size, device=device)).shape[1]
if feat_dim != 512:
    raise RuntimeError(f"Unexpected backbone feature dim: {feat_dim}. Expected 512 (non-ResNet50 setup).")
print(f"model params: {sum(p.numel() for p in net.parameters() if p.requires_grad):,}")

# projection head: 512 (existing backbone) -> 768 (CLIP ViT-L/14)
proj = nn.Sequential(
    nn.Linear(512, 1024),
    nn.GELU(),
    nn.Linear(1024, clip_feat_dim)
).to(device)

# train + val combined for distillation
ds1 = FaceFullSet(train_path, val_path, val_labels, tfm=aug_s1)
dl1 = DataLoader(ds1, batch_size=bsz, shuffle=True, drop_last=True, num_workers=n_workers)

# AdamW for CLIP feature alignment
opt1 = optim.AdamW(
    list(net.parameters()) + list(proj.parameters()),
    lr=1e-3, weight_decay=1e-4
)

num_ep_s1 = 35   # slightly fewer epochs
print(f"\nstarting CLIP distillation for {num_ep_s1} epochs...")

for ep in range(num_ep_s1):
    net.train()
    proj.train()

    cur_lr = lr_schedule(ep, 5, 1e-3, num_ep_s1)
    for g in opt1.param_groups:
        g['lr'] = cur_lr

    running_loss, n = 0.0, 0
    for imgs, _, idxs in dl1:
        imgs = imgs.to(device)
        targets = clip_feats[idxs].to(device)

        opt1.zero_grad()
        out = F.normalize(proj(net.extract_features(imgs)), dim=-1)
        loss = (1 - F.cosine_similarity(out, targets, dim=-1)).mean()
        loss.backward()

        nn.utils.clip_grad_norm_(list(net.parameters()) + list(proj.parameters()), 1.0)
        opt1.step()

        running_loss += loss.item() * imgs.size(0)
        n += imgs.size(0)

    print(f"ep {ep+1:03d}/{num_ep_s1} | lr: {cur_lr:.2e} | loss: {running_loss/n:.4f}")

# save distilled backbone
torch.save(net, backbone_file)
print(f"\ndone! saved backbone -> {backbone_file}")

# free memory
del clip_feats, proj, dl1, ds1
torch.cuda.empty_cache()

loading clip...
  train class 0: 9166 images
  train class 1: 9166 images
  val: 134 images
total features: torch.Size([18466, 768])
model params: 11,177,538
combined dataset: 18466 total images

starting distillation for 200 epochs...
ep 001/200 | loss: 0.2103
ep 002/200 | loss: 0.1802
ep 003/200 | loss: 0.1690
ep 004/200 | loss: 0.1597
ep 005/200 | loss: 0.1527
ep 006/200 | loss: 0.1477
ep 007/200 | loss: 0.1431
ep 008/200 | loss: 0.1394
ep 009/200 | loss: 0.1356
ep 010/200 | loss: 0.1324
ep 011/200 | loss: 0.1292
ep 012/200 | loss: 0.1265
ep 013/200 | loss: 0.1233
ep 014/200 | loss: 0.1209
ep 015/200 | loss: 0.1186
ep 016/200 | loss: 0.1161
ep 017/200 | loss: 0.1141
ep 018/200 | loss: 0.1113
ep 019/200 | loss: 0.1095
ep 020/200 | loss: 0.1076
ep 021/200 | loss: 0.1052
ep 022/200 | loss: 0.1032
ep 023/200 | loss: 0.1012
ep 024/200 | loss: 0.0993
ep 025/200 | loss: 0.0976
ep 026/200 | loss: 0.0958
ep 027/200 | loss: 0.0941
ep 028/200 | loss: 0.0923
ep 029/200 | loss: 0.0905
ep 030/200

## Stage 2: Fine-tune Existing Backbone for Age Classification

Fine-tune the CLIP-distilled backbone on **train + validation combined**.

Current setup:
- **30 epochs** with 5-epoch warmup
- **MixUp** (alpha=0.3) for first 75% of epochs
- **Exponential Moving Average (EMA)** of weights; final model uses EMA
- **AdamW** with weight_decay=1e-2
- Backbone LR: 5e-5 | Head LR: 5e-4
- RandAugment + RandomErasing in aug pipeline

In [ ]:
import copy

# train + val combined for stage 2
full_ds = FaceFullSet(train_path, val_path, val_labels, tfm=aug_s2)
full_dl = DataLoader(full_ds, batch_size=bsz, shuffle=True, drop_last=True, num_workers=n_workers)
print(f"training on {len(full_ds)} images (train + val)")

# differential LRs: lower for backbone (preserve distilled features), higher for head
head_p     = [p for n, p in net.named_parameters() if n.startswith("classifier")]
backbone_p = [p for n, p in net.named_parameters() if not n.startswith("classifier")]

opt2 = optim.AdamW([
    {"params": backbone_p, "lr": 5e-5},
    {"params": head_p,     "lr": 5e-4},
], weight_decay=1e-2)

loss_fn   = nn.CrossEntropyLoss(label_smoothing=0.05)  # reduced from 0.1
num_ep_s2 = 30

# EMA model - gives a smoother weight trajectory; use at final inference
ema_net   = copy.deepcopy(net)
ema_net.eval()
EMA_DECAY = 0.999   # slightly smoother than 0.998

print(f"finetuning for {num_ep_s2} epochs with MixUp + EMA...\n")

for ep in range(num_ep_s2):
    net.train()

    opt2.param_groups[0]["lr"] = lr_schedule(ep, 5, 5e-5, num_ep_s2)
    opt2.param_groups[1]["lr"] = lr_schedule(ep, 5, 5e-4, num_ep_s2)

    use_mixup = ep < (num_ep_s2 * 3 // 4)   # MixUp for first 75% epochs
    ep_loss, ep_correct, ep_total = 0.0, 0, 0

    for imgs, lbls, _ in full_dl:
        imgs, lbls = imgs.to(device), lbls.to(device)

        if use_mixup:
            imgs_m, y_a, y_b, lam = mixup_data(imgs, lbls, alpha=0.3)  # reduced from 0.4
        else:
            imgs_m, y_a = imgs, lbls

        opt2.zero_grad()
        logits = net(imgs_m)
        loss   = mixup_loss(loss_fn, logits, y_a, y_b, lam) if use_mixup else loss_fn(logits, y_a)
        loss.backward()
        nn.utils.clip_grad_norm_(net.parameters(), 1.0)
        opt2.step()

        # update EMA weights
        with torch.no_grad():
            for ep_p, net_p in zip(ema_net.parameters(), net.parameters()):
                ep_p.data.mul_(EMA_DECAY).add_(net_p.data, alpha=1.0 - EMA_DECAY)

        ep_loss    += loss.item() * imgs.size(0)
        ep_correct += (logits.argmax(1) == y_a).sum().item()
        ep_total   += imgs.size(0)

    t_acc = 100 * ep_correct / ep_total
    cur_lr = opt2.param_groups[0]["lr"]
    print(f"ep {ep+1:03d}/{num_ep_s2} | lr: {cur_lr:.2e} | loss: {ep_loss/ep_total:.4f} | acc: {t_acc:.1f}%")

# swap in EMA weights as the final model
net.load_state_dict(ema_net.state_dict())
print("\nfinetuning done! (swapped in EMA weights)")

combined dataset: 18466 total images
training on 18466 images (train + val)
finetuning for 50 epochs...



ep 01/50 | loss: 0.2443 | acc: 98.3%
ep 02/50 | loss: 0.2172 | acc: 99.4%
ep 03/50 | loss: 0.2137 | acc: 99.5%
ep 04/50 | loss: 0.2133 | acc: 99.4%
ep 05/50 | loss: 0.2122 | acc: 99.5%
ep 06/50 | loss: 0.2119 | acc: 99.6%
ep 07/50 | loss: 0.2117 | acc: 99.5%
ep 08/50 | loss: 0.2125 | acc: 99.5%
ep 09/50 | loss: 0.2111 | acc: 99.5%
ep 10/50 | loss: 0.2104 | acc: 99.6%
ep 11/50 | loss: 0.2105 | acc: 99.5%
ep 12/50 | loss: 0.2116 | acc: 99.5%
ep 13/50 | loss: 0.2111 | acc: 99.6%
ep 14/50 | loss: 0.2109 | acc: 99.5%
ep 15/50 | loss: 0.2101 | acc: 99.6%
ep 16/50 | loss: 0.2126 | acc: 99.4%
ep 17/50 | loss: 0.2114 | acc: 99.5%
ep 18/50 | loss: 0.2101 | acc: 99.6%
ep 19/50 | loss: 0.2103 | acc: 99.6%
ep 20/50 | loss: 0.2098 | acc: 99.6%
ep 21/50 | loss: 0.2096 | acc: 99.6%
ep 22/50 | loss: 0.2077 | acc: 99.7%
ep 23/50 | loss: 0.2087 | acc: 99.6%
ep 24/50 | loss: 0.2079 | acc: 99.7%
ep 25/50 | loss: 0.2061 | acc: 99.8%
ep 26/50 | loss: 0.2075 | acc: 99.7%
ep 27/50 | loss: 0.2086 | acc: 99.7%
e

In [ ]:
# save final model (EMA weights, existing backbone)
torch.save(net, "b23bb1025.pth")
print(f"saved b23bb1025.pth ({os.path.getsize('b23bb1025.pth') / 1e6:.1f} MB)")

# quick TTA check on validation set (horizontal-flip ensemble)
val_tfm = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_tfm_flip = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.RandomHorizontalFlip(p=1.0),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class ValDS(Dataset):
    def __init__(self, val_dir, val_csv):
        with open(val_csv) as f:
            self.items = [(r["image"], int(r["label"])) for r in csv.DictReader(f)]
        self.val_dir = val_dir
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        fn, lbl = self.items[i]
        img = Image.open(os.path.join(self.val_dir, fn)).convert("RGB")
        return val_tfm(img), val_tfm_flip(img), lbl

val_ds  = ValDS(val_path, val_labels)
val_dl  = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=n_workers)

net.eval()
correct, total = 0, 0
with torch.no_grad():
    for img, img_f, lbl in val_dl:
        img, img_f, lbl = img.to(device), img_f.to(device), lbl.to(device)
        logits_tta = (net(img) + net(img_f)) / 2   # TTA: average orig + hflip
        correct += (logits_tta.argmax(1) == lbl).sum().item()
        total   += lbl.size(0)

print(f"Validation accuracy (TTA) : {100 * correct / total:.2f}%")
print("submit: b23bb1025.py + b23bb1025.pth")

saved b23bb1025.pth (45.4 MB)
submit: b23bb1025.py + b23bb1025.pth + b23bb1025.pdf
